In [68]:
import pandas as pd
import xlrd
#import arcpy
import openpyxl
from pandas import ExcelWriter
from datetime import datetime
import os
#from arcgis.gis import GIS
#from arcgis.features import SpatialDataFrame
from shutil import copyfile
from openpyxl.utils.cell import get_column_letter
import urllib.request, json
#import new_geopoll_functions
import geopoll_modified
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

In [70]:
###USER DEFINED INPUTS BEGINNING
template_version = "20230720"
iso3 = "ZWE"
round_number = "11" #Specify the round number
admin_level = "Admin 2"
validation_number = 1 #For now this is not being used, keep it as 1 ; Please enter 1 or 2 ONLY
user_name = "Ibrahim" #Please write your name
country_questionnaire_file = r'GeoPoll_FAO_ZWE_EN_household_questionnaire_R11_V4.xlsx' #the questionnaire file that we use for creating the table

previousround_questionnaire = "yes" #yes if questionnaire built based on previous questionnaire validated; no if questionnaire built from scratch using template
previousround_questionnaire_file = r'validated_questionnaire_geopoll_en_ZWE_20240816123659.xlsx' #the previous validated questionnaire file that we use for creating the current one

## Modfiy path to respiratorycd
template_questionnaire_location = r'/Users/ibrahimaliahmad/Library/Mobile Documents/com~apple~CloudDocs/Documents/FAO/Data in Emergencies (DIEM)/Working Files/Questionnaire Validation/Script/questionnaire_validation/Questionnaire Validation/Questionnaires/Questionnaire Templates'
previousround_questionnaire_location = r'/Users/ibrahimaliahmad/Library/Mobile Documents/com~apple~CloudDocs/Documents/FAO/Data in Emergencies (DIEM)/Working Files/Questionnaire Validation/Script/questionnaire_validation/Questionnaire Validation/Questionnaires/Questionnaire Previous Round'
country_questionnaires_location = r'/Users/ibrahimaliahmad/Library/Mobile Documents/com~apple~CloudDocs/Documents/FAO/Data in Emergencies (DIEM)/Working Files/Questionnaire Validation/Script/questionnaire_validation/Questionnaire Validation/Questionnaires/Questionnaire Country'
output_locations = r'/Users/ibrahimaliahmad/Library/Mobile Documents/com~apple~CloudDocs/Documents/FAO/Data in Emergencies (DIEM)/Working Files/Questionnaire Validation/Script/questionnaire_validation/Questionnaire Validation/Questionnaires/Validated Questionnaires/ZWE/R11/V2'
##

###USER DEFINED INPUTS END

In [72]:
# Call the detect_enumerator function from the geopoll_modified module, using the country_questionnaire_file as an argument to determine the enumerator type (either "geopoll" or "kobo").
enumerator = geopoll_modified.detect_enumerator(country_questionnaire_file)

# Call the detect_language function from the geopoll_modified module, using the country_questionnaire_file to detect the language used in the file (e.g., "en", "fr", "es", "ar").
lang = geopoll_modified.detect_language(country_questionnaire_file)

# Capture the current date and time using the datetime module.
startTime = datetime.now()
now = datetime.now().strftime('%Y%m%d%H%M%S')

# Import the datetime class from the datetime module.
from datetime import datetime
now1 = datetime.now()
formatted_now = now1.strftime("%d/%m/%Y, %H:%M:%S")

# Initialize a string to keep track of various execution messages.
execution_messages = ''
execution_messages += "\n%s%s" % ("Country iso3: ", iso3)
execution_messages += "\n%s%s" % ("Country round #: ", round_number)
execution_messages += "\n%s%s" % ("Country Questionnaire Language: ", lang)
execution_messages += "\n%s%s" % ("Country Questionnaire Type: ", enumerator)
execution_messages += "\n%s%s" % ("Country Questionnaire File: ", country_questionnaire_file)


# Concatenate the path for the country questionnaire file.
country_questionnaire_file = os.path.join(country_questionnaires_location, country_questionnaire_file)

# Call detect_template function (not provided in the pasted code) to get the template questionnaire file, then concatenate the path.

template_questionnaire_file = geopoll_modified.detect_template(template_version,country_questionnaire_file)
template_questionnaire_file = os.path.join(template_questionnaire_location, template_questionnaire_file)
execution_messages += "\n%s%s" % ("The template used to validate the country questionnaire is: ", template_version)   


#template_questionnaire_file = os.path.join(r'C:/temp', 'validated_questionnaire_geopoll_es_HON_20231128115537_v2.xlsx')
#template_questionnaire_file = os.path.join(template_questionnaire_location, template_questionnaire_file)
#execution_messages += "\n%s%s" % ("The template used to validate the country questionnaire is: ", template_version)   



# If there is a previous round questionnaire, concatenate its path and add the information to the execution messages.
if previousround_questionnaire == "yes":
    execution_messages += "\n%s%s" % ("Previous Questionnaire used: ", previousround_questionnaire_file)   
    previousround_questionnaire_file = os.path.join(previousround_questionnaire_location, previousround_questionnaire_file)

# Continue building the execution_messages string with further details like validation number, user name, and the date and time of validation.
execution_messages += "\n%s%s" % ("Validation number: ", validation_number)
execution_messages += "\n%s%s" % ("User: ", user_name)
execution_messages += "\n%s%s" % ("Validated on: ", formatted_now)

# Print the execution messages, summarizing the execution details.
print(execution_messages)


Country iso3: ZWE
Country round #: 11
Country Questionnaire Language: en
Country Questionnaire Type: geopoll
Country Questionnaire File: GeoPoll_FAO_ZWE_EN_household_questionnaire_R11_V4.xlsx
The template used to validate the country questionnaire is: 20230720
Previous Questionnaire used: validated_questionnaire_geopoll_en_ZWE_20240816123659.xlsx
Validation number: 1
User: Ibrahim
Validated on: 18/07/2025, 13:24:11


In [74]:
##CHECK THE QUESTIONNAIRE BEFORE PROCEEDING TO THE NEXT CELL##
print("Checking the conditionality of all questions")

# Define the file path for the conditionality questionnaire
questions_conditionality = os.path.join(output_locations, "conditionality_questionnaire_%s_%s_%s_%s.xlsx" % (enumerator,lang,iso3, now)) 

# Call the check_all_questions function from the geopoll_modified module
# This function compares two Excel files (country_questionnaire_file and template_questionnaire_file) and returns brief and detailed results
conditionality_result_brief, conditionality_result_details = geopoll_modified.check_all_questions(country_questionnaire_file, template_questionnaire_file, questions_conditionality)

print(conditionality_result_brief)

# Add the detailed result to the execution_messages variable
execution_messages += "\n\n" + conditionality_result_details

Checking the conditionality of all questions

detecting differences in column: Suggested Name

detecting differences in column: English

detecting differences in column: Length

detecting differences in column: Q Type

detecting differences in column: Conditional

detecting differences in column: Programming Instructions

detecting differences in column: Skip Pattern

detecting differences in column: Codes

detecting differences in column: Default skip patterns & conditional 

detecting differences in column: Mandatory

detecting differences in column: Core questions only



In [75]:
# Printing an informative message about the current process
print("Reading TEMPLATE questionnaire and creating recap excel file")

# Assigning the template questionnaire file to a variable
questionnaire_file = template_questionnaire_file

# Assigning the output location to a temporary path variable
temp_path = output_locations

# Creating an intermediary output file name with a timestamp
coded_values_file = os.path.join(temp_path, "coded_values_%s_template.xlsx" % now)

# Initializing a list to hold field names
field_names_list = []

# Limiting the execution to a maximum number of items (for testing purposes)
max_counter = 6000

# Calling the read_questionnaire function and storing the returned languages
languages = geopoll_modified.read_questionnaire(questionnaire_file, coded_values_file)

# Repeat the process for COUNTRY questionnaire
print("Reading COUNTRY questionnaire and creating recap excel file")
country_coded_values_file = os.path.join(temp_path, "coded_values_%s_country.xlsx" % now)
field_names_list = []
max_counter = 6000
languages = geopoll_modified.read_questionnaire(country_questionnaire_file, country_coded_values_file)


Reading TEMPLATE questionnaire and creating recap excel file
Version in English only
Reading COUNTRY questionnaire and creating recap excel file
Version in English only


In [76]:
print ("COUNTING NUMBER OF QUESTIONS")

#T is for the Template Questionnaire
#C is for the Country Questionnaire
#P is the the 'Previous' Round Questionnaire

# Count the number of questions based on the "Suggested Qname" field for the template and country questionnaires
T_sqname_list_questions, T_sqname_n_of_questions = geopoll_modified.count_number_of_questions_sqname(questionnaire_file)
T_qname_list_questions, T_qname_n_of_questions = geopoll_modified.count_number_of_questions_qname(questionnaire_file)
C_sqname_list_questions, C_sqname_n_of_questions = geopoll_modified.count_number_of_questions_sqname(country_questionnaire_file)
C_qname_list_questions, C_qname_n_of_questions = geopoll_modified.count_number_of_questions_qname(country_questionnaire_file)

message1 = "\nTemplate Questionnaire: %s" % T_qname_n_of_questions
message2 = "Current Questionnaire: %s" % C_qname_n_of_questions

# If there was a questionnaire in a previous round, count the questions for that as well
if previousround_questionnaire == "yes":
    P_sqname_list_questions, P_sqname_n_of_questions = geopoll_modified.count_number_of_questions_sqname(previousround_questionnaire_file)
    P_qname_list_questions, P_qname_n_of_questions = geopoll_modified.count_number_of_questions_qname(previousround_questionnaire_file)
    message2P = "Previous Questionnaire: %s\n" % P_qname_n_of_questions

message3 = "\nTemplate Questionnaire: %s" % T_sqname_n_of_questions
message4 = "Current Questionnaire: %s" % C_sqname_n_of_questions

# Display the number of questions. Messages differ based on whether there was a previous round questionnaire.
if previousround_questionnaire == "yes":
    message4P = "Previous Questionnaire: %s" % P_sqname_n_of_questions

if previousround_questionnaire == "yes":
    execution_messages += "\n%s\n%s\n%s\n%s\n%s\n%s\n%s\n%s" % ("QName Field:", message1, message2, message2P, "SuggestedQName Field:", message3, message4, message4P)
    print("\nQName Field:")
    print(message1)
    print(message2)
    print(message2P)
    print("SuggestedQName Field:")
    print(message3)
    print(message4)
    print(message4P)

    
else:
    execution_messages += "\n\n%s\n%s\n%s\n%s\n%s\n%s" % ("QName Field:", message1, message2, "SuggestedQName Field:", message3, message4)
    print("QName Field:")
    print(message1)
    print(message2)
    print("SuggestedQName Field:")
    print(message3)
    print(message4)




COUNTING NUMBER OF QUESTIONS

QName Field:

Template Questionnaire: 178
Current Questionnaire: 258
Previous Questionnaire: 166

SuggestedQName Field:

Template Questionnaire: 176
Current Questionnaire: 150
Previous Questionnaire: 164


In [77]:
#OLD FUNCTION the next function is the one to be used

#print ("CHECKING IF ALL MANDATORY QUESTIONS ARE INCLUDED IN THE COUNTRY QUESTIONNAIRE")

# Load the questionnaire file from an excel file into a pandas DataFrame; we skip the first 2 rows as they are headers or metadata
#quest_df = pd.read_excel(open(questionnaire_file, 'rb'), sheet_name='survey',skiprows=2)

# From the DataFrame, get a list of mandatory question names that don't contain "hh_wealth_" in their name
# The list is derived from rows where "Q Name" doesn't contain "hh_wealth_" and "Mandatory" is set to "yes"
# Values of "Q Name" are converted to list and any leading/trailing whitespaces in each question name is removed

#list_mandatory_questions = quest_df.loc[((quest_df["Q Name"].str.contains("hh_wealth_") == False ) & (quest_df["Mandatory"] == "yes"))]["Q Name"].values.tolist() 
#list_mandatory_questions = [s.strip() for s in list_mandatory_questions]
#message9 = "\nFound %s mandatory questions in the template questionnaire" % len(list_mandatory_questions)
#print(message9)

# Check if all mandatory questions are present in the country questionnaire (C_qname_list_questions)
#result =  all(elem in C_qname_list_questions  for elem in list_mandatory_questions)
#if result:
#    message10 = "Yes, current questionnaire contains all mandatory questions"   
#else:
#    message10 = "No, current questionnaire does not contains all mandatory questions. Missing question: %s " % list(set(list_mandatory_questions ) - set(C_qname_list_questions))
#print(message10)
    
#execution_messages += "\n%s\n%s" % (message9, message10)

In [79]:
print("📋 CHECKING IF ALL MANDATORY QUESTIONS ARE INCLUDED IN THE COUNTRY QUESTIONNAIRE")

# Load mandatory questions from the template
quest_df = pd.read_excel(open(questionnaire_file, 'rb'), sheet_name='survey', skiprows=2)
list_mandatory_questions = quest_df.loc[
    (~quest_df["Q Name"].str.contains("hh_wealth_", na=False)) & (quest_df["Mandatory"] == "yes"),
    "Q Name"
].dropna().str.strip().tolist()

message9 = f"\nFound {len(list_mandatory_questions)} mandatory questions in the template questionnaire"
print(message9)

# Compare against Country questionnaire
missing_vs_country = list(set(list_mandatory_questions) - set(C_qname_list_questions))
status_country = "✅ All Present" if not missing_vs_country else "❌ Not All Found"

# Compare against Previous Round
if previousround_questionnaire == "yes":
    missing_vs_previous = list(set(list_mandatory_questions) - set(P_qname_list_questions))
    status_previous = "✅ All Present" if not missing_vs_previous else "❌ Not All Found"
else:
    missing_vs_previous = []
    status_previous = "N/A"

# Create table-style summary
message_summary = "\n📊 Mandatory Questions Check\n"
message_summary += "| Compared To     | Status         | Missing Count | Action                                  |\n"
message_summary += "|-----------------|----------------|----------------|------------------------------------------|\n"
message_summary += f"| Template         | {status_country:<15} | {len(missing_vs_country):<14} | {'Review missing questions' if missing_vs_country else 'No action needed'} |\n"
message_summary += f"| Previous Round   | {status_previous:<15} | {len(missing_vs_previous):<14} | {'Review if intentionally dropped' if missing_vs_previous else 'No action needed'} |\n"

# Add missing question details
details = "\n🧾 Missing Mandatory Questions:\n"
if missing_vs_country:
    details += f"\nFrom Template (vs. Country):\n- " + "\n- ".join(sorted(missing_vs_country))
if previousround_questionnaire == "yes" and missing_vs_previous:
    details += f"\n\nFrom Template (vs. Previous Round):\n- " + "\n- ".join(sorted(missing_vs_previous))
if not missing_vs_country and not missing_vs_previous:
    details += "None — all mandatory questions are present."

# Output
print(message_summary)
print(details)

execution_messages += message9 + "\n" + message_summary + "\n" + details


📋 CHECKING IF ALL MANDATORY QUESTIONS ARE INCLUDED IN THE COUNTRY QUESTIONNAIRE

Found 81 mandatory questions in the template questionnaire

📊 Mandatory Questions Check
| Compared To     | Status         | Missing Count | Action                                  |
|-----------------|----------------|----------------|------------------------------------------|
| Template         | ❌ Not All Found | 1              | Review missing questions |
| Previous Round   | ✅ All Present   | 0              | No action needed |


🧾 Missing Mandatory Questions:

From Template (vs. Country):
- hh_admin2_1


In [80]:
#OLD Function
#Work on the output messages aka/ execution messages should be clearer to also include action points (severity level).
# We can keep the differences results between templates and current questionnaire but store it in the detailed report only
#Renzi: Maybe, also include actions point/s for each message. What do we want the reader of the output/reader to do with the finding? 


#print ("DETECTING DIFFERENCES IN FIELD: Suggested QName")

# Calculate the questions which are unique to the country questionnaire
#unique_country_questionnaire = list(set(C_sqname_list_questions) - set(T_sqname_list_questions))
# Calculate the questions which are unique to the template questionnaire
#unique_template_questionnaire = list(set(T_sqname_list_questions) - set(C_sqname_list_questions))

#Calculate the questions which are unique to the Previous Round questionnaire
#if previousround_questionnaire == "yes":
#    unique_previousround_questionnaire = list(set(C_sqname_list_questions) - set(P_sqname_list_questions))

# Construct the message to show questions that are only in the country questionnaire
#message5 = "\nSuggestedQName Field - Questions in country questionnaire only (therefore not stored/published on the Hub):"
#for question in unique_country_questionnaire:
#    message5 += f"\n- {question}"

# Construct the message to show questions that are only in the template questionnaire
#message6 = "\nSuggestedQName Field - Questions in template questionnaire only (therefore not collected in the country):"
#for question in unique_template_questionnaire:
#    message6 += f"\n- {question}"

#if previousround_questionnaire == "yes":
    # Construct the message to show questions that are only in the template questionnaire
#    message06 = "\nSuggestedQName Field - Questions in Previous questionnaire only, compared to new Round (therefore not collected in the country):"
#    for question in unique_previousround_questionnaire:
#        message06 += f"\n- {question}"

# Append both messages to the overall execution_messages
#execution_messages += "\n%s\n%s" % (message5, message6)

# Call the function to highlight the differences between the two files when we are comparing with a previous questionnaire
#geopoll_modified.highlight_differences_in_qname(country_questionnaire_file, previousround_questionnaire_file)

#if previousround_questionnaire == "yes":
    # Append both messages to the overall execution_messages
#    execution_messages += "\n%s\n%s" % (message5, message06)

    # Call the function to highlight the differences between the two files when we are comparing with a previous questionnaire
#    geopoll_modified.highlight_differences_in_qname(country_questionnaire_file, previousround_questionnaire_file)

#print (message5)
#print (message6)
#if previousround_questionnaire == "yes":
#    print (message06)

In [82]:
print("📌 DETECTING DIFFERENCES IN FIELD: Suggested QName")

# Difference calculation
unique_country_questionnaire = list(set(C_sqname_list_questions) - set(T_sqname_list_questions))
unique_template_questionnaire = list(set(T_sqname_list_questions) - set(C_sqname_list_questions))

if previousround_questionnaire == "yes":
    unique_previousround_questionnaire = list(set(P_sqname_list_questions) - set(C_sqname_list_questions))

# Results summary table
summary_table = "\n📊 SuggestedQName Differences Summary\n"
summary_table += "| Compared To       | Status             | Difference Count | Severity | Action                         |\n"
summary_table += "|-------------------|---------------------|------------------|----------|--------------------------------|\n"

# Template comparison
template_diff_count = len(unique_template_questionnaire)
template_status = "✅ No Differences" if template_diff_count == 0 else "❌ Differences Found"
template_severity = "Low" if template_diff_count == 0 else "Medium"
template_action = "No action needed" if template_diff_count == 0 else "Review added/missing questions"
summary_table += f"| Template           | {template_status:<19} | {template_diff_count:<16} | {template_severity:<8} | {template_action:<30} |\n"

# Previous round comparison
if previousround_questionnaire == "yes":
    previous_diff_count = len(unique_previousround_questionnaire)
    previous_status = "✅ No Differences" if previous_diff_count == 0 else "❌ Differences Found"
    previous_severity = "Low" if previous_diff_count == 0 else "Medium"
    previous_action = "No action needed" if previous_diff_count == 0 else "Check removed questions from new round"
    summary_table += f"| Previous Round     | {previous_status:<19} | {previous_diff_count:<16} | {previous_severity:<8} | {previous_action:<30} |\n"

# Country-only questions (not on hub)
country_diff_count = len(unique_country_questionnaire)
country_status = "✅ All Published" if country_diff_count == 0 else "⚠️ Not Published"
country_severity = "Low" if country_diff_count == 0 else "Low"
country_action = "No action needed" if country_diff_count == 0 else "Validate if should be on the hub"
summary_table += f"| Country Only (Hub) | {country_status:<19} | {country_diff_count:<16} | {country_severity:<8} | {country_action:<30} |\n"

# Append to execution messages
execution_messages += summary_table

# Save detailed differences to report (example structure, update this with actual logging logic)
detailed_differences = {
    "Questions only in country (not on hub)": unique_country_questionnaire,
    "Questions only in template (not in country)": unique_template_questionnaire
}
if previousround_questionnaire == "yes":
    detailed_differences["Questions only in previous round"] = unique_previousround_questionnaire

# Optional: Store this into Excel or text report
# save_to_report(detailed_differences, 'suggested_qname_differences_report.xlsx')

# Print summary table only
print(summary_table)

# Call visual diff (if applicable)
if previousround_questionnaire == "yes":
    geopoll_modified.highlight_differences_in_qname(country_questionnaire_file, previousround_questionnaire_file)



# Optional flag to control detailed printouts
show_differences = True  # Set to False to suppress all raw outputs
diff_threshold = 25      # Minimum count to auto-display differences

# Append detailed findings to execution messages
if show_differences or template_diff_count > diff_threshold:
    execution_messages += "\n\n📝 Questions in Template but Missing in Country Questionnaire:\n"
    print("\n📝 Questions in Template but Missing in Country Questionnaire:")
    for q in unique_template_questionnaire:
        line = f"- {q}\n"
        execution_messages += line
        print(line.strip())

if previousround_questionnaire == "yes" and (show_differences or previous_diff_count > diff_threshold):
    execution_messages += "\n\n📝 Questions in Previous Round but Missing in Current Round:\n"
    print("\n📝 Questions in Previous Round but Missing in Current Round:")
    for q in unique_previousround_questionnaire:
        line = f"- {q}\n"
        execution_messages += line
        print(line.strip())

if show_differences or country_diff_count > diff_threshold:
    execution_messages += "\n\n📝 Questions in Country Questionnaire not in Template (i.e., not published on the Hub):\n"
    print("\n📝 Questions in Country Questionnaire not in Template (i.e., not published on the Hub):")
    for q in unique_country_questionnaire:
        line = f"- {q}\n"
        execution_messages += line
        print(line.strip())


📌 DETECTING DIFFERENCES IN FIELD: Suggested QName

📊 SuggestedQName Differences Summary
| Compared To       | Status             | Difference Count | Severity | Action                         |
|-------------------|---------------------|------------------|----------|--------------------------------|
| Template           | ❌ Differences Found | 49               | Medium   | Review added/missing questions |
| Previous Round     | ❌ Differences Found | 31               | Medium   | Check removed questions from new round |
| Country Only (Hub) | ⚠️ Not Published    | 23               | Low      | Validate if should be on the hub |

✅ Highlighted English column, Suggested Qname, and Q Name (by lookup): Green = same, Yellow = changed.

📝 Questions in Template but Missing in Country Questionnaire:
- cs_stress_borrowed_or_helped
- cs_emergency_sold_house
- cs_stress_spent_savings
- hh_admin2_1
- hh_asset_transp_

1) hh_asset_transp_bici
2) hh_asset_transp_cart
3) hh_asset_transp_boatmotor
4) h

In [83]:
#print ("DETECTING DIFFERENCES IN FIELD: Q Name")

# Calculate the questions which are unique to the country questionnaire
#unique_country_questionnaire_qname = list(set(C_qname_list_questions) - set(T_qname_list_questions))

# Calculate the questions which are unique to the template questionnaire
#unique_template_questionnaire_qname = list(set(T_qname_list_questions) - set(C_qname_list_questions))

#Calculate the questions which are unique to the Previous Round questionnaire
#if previousround_questionnaire == "yes":
#    unique_previousround_questionnaire_qname = list(set(C_qname_list_questions) - set(P_qname_list_questions))

# Construct the message to show questions that are only in the country questionnaire
#message7 = "\nQName Field - Questions in country questionnaire only (therefore not stored/published on the Hub):"
#for question in unique_country_questionnaire_qname:
#    message7 += f"\n- {question}"

# Construct the message to show questions that are only in the template questionnaire
#message8 = "\nQName Field - Questions in template questionnaire only (therefore not collected in the country):"
#for question in unique_template_questionnaire_qname:
#    message8 += f"\n- {question}"
    
#if previousround_questionnaire == "yes":
    # Construct the message to show questions that are only in the template questionnaire
#    message08 = "\nSuggestedQName Field - Questions in Previous questionnaire only, compared to new Round (therefore not collected in the country):"
#    for question in unique_previousround_questionnaire_qname:
#        message08 += f"\n- {question}"

# Append both messages to the overall execution_messages
#execution_messages += "\n%s\n%s" % (message7, message8)

#if previousround_questionnaire == "yes":
#    # Append both messages to the overall execution_messages
#    execution_messages += "\n%s\n%s" % (message7, message08)

    # Call the function to highlight the differences between the two files when we are comparing with a previous questionnaire
#    geopoll_modified.highlight_differences_in_qname(country_questionnaire_file, previousround_questionnaire_file)

# Call the function to highlight the differences between the two files when we are comparing with a previous questionnaire
#print (message7)
#print (message8)
#if previousround_questionnaire == "yes":
#    print (message08)

In [84]:
print("📌 DETECTING DIFFERENCES IN FIELD: Q Name")

# Calculate differences
unique_country_questionnaire_qname = list(set(C_qname_list_questions) - set(T_qname_list_questions))
unique_template_questionnaire_qname = list(set(T_qname_list_questions) - set(C_qname_list_questions))

if previousround_questionnaire == "yes":
    unique_previousround_questionnaire_qname = list(set(P_qname_list_questions) - set(C_qname_list_questions))

# Summary Table
qname_summary = "\n📊 Q Name Differences Summary\n"
qname_summary += "| Compared To       | Status             | Difference Count | Severity | Action                         |\n"
qname_summary += "|-------------------|---------------------|------------------|----------|--------------------------------|\n"

# Template
qname_template_diff_count = len(unique_template_questionnaire_qname)
qname_template_status = "✅ No Differences" if qname_template_diff_count == 0 else "❌ Differences Found"
qname_template_severity = "Low" if qname_template_diff_count == 0 else "Medium"
qname_template_action = "No action needed" if qname_template_diff_count == 0 else "Review added/missing questions"
qname_summary += f"| Template           | {qname_template_status:<19} | {qname_template_diff_count:<16} | {qname_template_severity:<8} | {qname_template_action:<30} |\n"

# Previous Round
if previousround_questionnaire == "yes":
    qname_previous_diff_count = len(unique_previousround_questionnaire_qname)
    qname_previous_status = "✅ No Differences" if qname_previous_diff_count == 0 else "❌ Differences Found"
    qname_previous_severity = "Low" if qname_previous_diff_count == 0 else "Medium"
    qname_previous_action = "No action needed" if qname_previous_diff_count == 0 else "Check removed questions from new round"
    qname_summary += f"| Previous Round     | {qname_previous_status:<19} | {qname_previous_diff_count:<16} | {qname_previous_severity:<8} | {qname_previous_action:<30} |\n"

# Country Only
qname_country_diff_count = len(unique_country_questionnaire_qname)
qname_country_status = "✅ All Published" if qname_country_diff_count == 0 else "⚠️ Not Published"
qname_country_severity = "Low" if qname_country_diff_count == 0 else "Low"
qname_country_action = "No action needed" if qname_country_diff_count == 0 else "Validate if should be on the hub"
qname_summary += f"| Country Only (Hub) | {qname_country_status:<19} | {qname_country_diff_count:<16} | {qname_country_severity:<8} | {qname_country_action:<30} |\n"

# Append summary
execution_messages += qname_summary
print(qname_summary)

# Detailed outputs and logging
if show_differences or qname_template_diff_count > diff_threshold:
    section = "\n📝 Q Name in Template but Missing in Country Questionnaire:\n"
    execution_messages += section
    print(section.strip())
    for q in unique_template_questionnaire_qname:
        execution_messages += f"- {q}\n"
        print(f"- {q}")

if previousround_questionnaire == "yes" and (show_differences or qname_previous_diff_count > diff_threshold):
    section = "\n📝 Q Name in Previous Round but Missing in Current Round:\n"
    execution_messages += section
    print(section.strip())
    for q in unique_previousround_questionnaire_qname:
        execution_messages += f"- {q}\n"
        print(f"- {q}")

if show_differences or qname_country_diff_count > diff_threshold:
    section = "\n📝 Q Name in Country Questionnaire not in Template (i.e., not published on the Hub):\n"
    execution_messages += section
    print(section.strip())
    for q in unique_country_questionnaire_qname:
        execution_messages += f"- {q}\n"
        print(f"- {q}")

# Visual diff (optional, depends on Q Name relevance)
if previousround_questionnaire == "yes":
    geopoll_modified.highlight_differences_in_qname(
        country_questionnaire_file,
        previousround_questionnaire_file
    )

📌 DETECTING DIFFERENCES IN FIELD: Q Name

📊 Q Name Differences Summary
| Compared To       | Status             | Difference Count | Severity | Action                         |
|-------------------|---------------------|------------------|----------|--------------------------------|
| Template           | ❌ Differences Found | 38               | Medium   | Review added/missing questions |
| Previous Round     | ❌ Differences Found | 21               | Medium   | Check removed questions from new round |
| Country Only (Hub) | ⚠️ Not Published    | 118              | Low      | Validate if should be on the hub |

📝 Q Name in Template but Missing in Country Questionnaire:
- cs_stress_borrowed_or_helped
- cs_emergency_sold_house
- cs_stress_spent_savings
- hh_admin2_1
- covid
- rcsi_restrict_adult_consumpt
- fcs_sugar_days
- crp_seed_supply
- fcs_staple_days
- fcs_oil_days
- fcs_vegetables_days
- hh_wealth_toilet
- crp_harv_lastyr
- crp_harv_unit_kg
- crp_harv_vol
- ls_all
- cs_crisis_redu

In [85]:
#We should have at least 1 wealth question. Respected > 1 and no Maximum!

# Print the header indicating the field being checked
print ("CHECKING THAT RULES ON HH_WEALTH_ QUESTIONS ARE RESPECTED\n")

# Extract all questions from the country questionnaire that start with 'hh_wealth_'
wealth_questions_in_country = [i for i in C_qname_list_questions if i.startswith('hh_wealth_')]
o_wealth_questions_in_country = [j for j in C_qname_list_questions if j.startswith('o_hh_wealth')]


# Check if there's only one 'hh_wealth_' question in the country questionnaire
if len(wealth_questions_in_country) == 1:
    # Construct a message to indicate that the rule has been respected
    message11 = "\n1 hh_wealth_ question found in the country questionnaire: approved\n"

 # Construct a message to indicate that there are too many 'hh_wealth_' questions
else:
    message11 = message11 = "\n%s hh_wealth_ questions found in the country questionnaire: it should be only one!\n" % len(wealth_questions_in_country)

# Check if there's only one 'o_hh_wealth_' question in the country questionnaire
if len(o_wealth_questions_in_country) == 1:
    # Construct a message to indicate that the rule has been respected
    message11a = "\n1 o_hh_wealth_ question found in the country questionnaire: approved\n"
elif len(o_wealth_questions_in_country) == 0:
    message11a = message11a = ""
 # Construct a message to indicate that there are too many 'hh_wealth_' questions
else:
    message11a = message11a = "\n%s o_hh_wealth_ questions found in the country questionnaire\n" % len(o_wealth_questions_in_country)



# Print the result for the wealth questions check
print(message11)
print(message11a)


# Append the message to the overall execution_messages
execution_messages += "\n%s" % (message11)
execution_messages += "\n%s" % (message11a)

CHECKING THAT RULES ON HH_WEALTH_ QUESTIONS ARE RESPECTED


1 hh_wealth_ question found in the country questionnaire: approved




In [86]:
# Print header indicating the section's purpose
print ("CHECKING THAT RULES ON CS QUESTIONS ARE RESPECTED\n")

# Extract all questions in the country questionnaire starting with 'cs'
cs_questions_in_country = [i for i in C_qname_list_questions if i.startswith('cs')]

# If there are any 'cs' questions in the country questionnaire
if len(cs_questions_in_country) > 0:

    # Extract all questions in the country questionnaire starting with 'cs_stress'
    cs_stress_questions_in_country = [i for i in C_qname_list_questions if i.startswith('cs_stress')];
    
    # Extract all questions in the country questionnaire starting with 'cs_emergency'
    cs_emergency_questions_in_country = [i for i in C_qname_list_questions if i.startswith('cs_emergency')];
    
    # Extract all questions in the country questionnaire starting with 'cs_crisis'
    cs_crisis_questions_in_country = [i for i in C_qname_list_questions if i.startswith('cs_crisis')];
    
    # Check if the total number of each category of 'cs' questions meets the specified conditions
    if (len(cs_stress_questions_in_country) >= 4) & (len(cs_emergency_questions_in_country)>=3) & (len(cs_crisis_questions_in_country) >= 3):
        
        # Construct message indicating rules on CS questions are respected
        message111 = "\n%s CS questions found in the country questionnaire\n" % len(cs_questions_in_country) + "There are more than 10 coping strategies included in the questionnaire. Make sure that each respondent profile will be administered at least 10 coping strategies.\nNumber of STRESS questions: %s" % len(cs_stress_questions_in_country) + "\nNumber of CRISIS questions: %s\n" % len(cs_crisis_questions_in_country) + "Number of EMERGENCY questions: %s\n" % len(cs_emergency_questions_in_country) + "Rule respected"
        
    else: 
        
        # Construct message indicating rules on CS questions are not respected
        message111 = "\n%s CS questions found in the country questionnaire\n" % len(cs_questions_in_country) + "There are less than 10 coping strategies included in the questionnaire. Add coping strategies so that each respondent profile receives at least 10 coping strategies.\nNumber of STRESS questions: %s" % len(cs_stress_questions_in_country) + "\nNumber of CRISIS questions: %s\n" % len(cs_crisis_questions_in_country) + "Number of EMERGENCY questions: %s\n" % len(cs_emergency_questions_in_country) + "Rule NOT respected"

else:

    # Construct message when no CS questions are found in the country questionnaire
    message111 = "\nNo CS questions in the country questionnaire.\nThere are less than 10 coping strategies included in the questionnaire. Add coping strategies so that each respondent profile receives at least 10 coping strategies."

# Print the message regarding the evaluation of rules on CS questions
print(message111)

# Append the message to the overall execution_messages for further processing or logging
execution_messages += "\n%s" % (message111)

CHECKING THAT RULES ON CS QUESTIONS ARE RESPECTED


10 CS questions found in the country questionnaire
There are more than 10 coping strategies included in the questionnaire. Make sure that each respondent profile will be administered at least 10 coping strategies.
Number of STRESS questions: 4
Number of CRISIS questions: 3
Number of EMERGENCY questions: 3
Rule respected


In [87]:
# Print a message indicating that the code will check rules for HDDS questions
print ("CHECKING THAT RULES ON HDDS QUESTIONS ARE RESPECTED")

# Get a list of questions from the country questionnaire that start with 'hdds'
hdds_questions_in_country = [i for i in C_qname_list_questions if i.startswith('hdds')]

# Initial message indicating the general rule about HDDS fields
message12 = "\nThere should be more than 2 HDDS fields"

# Check if there are any HDDS questions in the country questionnaire
if len(hdds_questions_in_country) > 0:
    # Append to the message the number of HDDS questions found and their names
    message12 += "\nNumber of HDDS questions: %s  - %s" % (len(hdds_questions_in_country), hdds_questions_in_country)
else:
    # Append to the message that there are no HDDS questions in the country questionnaire
    message12 += "\nNo HDDS questions in the country questionnaire"

# Append the constructed message about HDDS questions to the overall execution_messages
execution_messages += "\n%s" % (message12)

# Print the constructed message about HDDS questions
print(message12)

CHECKING THAT RULES ON HDDS QUESTIONS ARE RESPECTED

There should be more than 2 HDDS fields
Number of HDDS questions: 2  - ['hdds', 'hdds_confirmation']


In [90]:
#DOUBLE CHECK WHERE AS WE NEED TO INCLUDE Either ALL questions including the optional ones (#4) or either only the mandatory 1

# Display an informative message for the user
print ("CHECKING THAT RULES ON CROP HARVEST QUESTIONS ARE RESPECTED")

# Extract questions related to 'crp_harv' from the country questionnaire, but exclude 'crp_harv_lastyr'
crp_questions_in_country = [i for i in C_qname_list_questions if (i.startswith('crp_harv') and i != ('crp_harv_lastyr'))]

# Begin constructing the message to inform about the rules of the "crp_harv" questions
message09 = "\nThere should be either only \"crp_harv_change\" or the 4 crp_harv questions from the template: "

# Check if the country questionnaire has exactly 4 'crp_harv' questions and also includes 'crp_harv_change'. If so, rule is respected.
if (len(crp_questions_in_country) == 4) and ("crp_harv_change" in crp_questions_in_country):
    message09 += "RULE IS RESPECTED\nNumber of crp_harv questions: %s  - %s" % (len(crp_questions_in_country), crp_questions_in_country)

# Check if the country questionnaire has only 'crp_harv_change'. If so, rule is respected.
elif (len(crp_questions_in_country) == 1) and ("crp_harv_change" in crp_questions_in_country):
    message09 += "RULE IS RESPECTED\nNumber of crp_harv questions: %s  - %s" % (len(crp_questions_in_country), crp_questions_in_country)

# If none of the above conditions are met, then the rule is not respected.
else:
    message09 += "RULE IS NOT RESPECTED\nNumber of crp_harv questions: %s  - %s" % (len(crp_questions_in_country), crp_questions_in_country)

# Append the message to the overall 'execution_messages' to log the results
execution_messages += "\n%s" % (message09)

# Print the constructed message to show the results of the check
print(message09)

CHECKING THAT RULES ON CROP HARVEST QUESTIONS ARE RESPECTED

There should be either only "crp_harv_change" or the 4 crp_harv questions from the template: RULE IS RESPECTED
Number of crp_harv questions: 1  - ['crp_harv_change']


In [93]:
#make it based on the previous questionnaire only!!!!

import pandas as pd

print("Comparison of Domains - Marked Change from Template\n")

# Function to compare domains between two dataframes for a specific column, with enhanced output formatting
def compare_and_print_differences(df1, df2, column_index, comparison_type="Template"):
    for index, row in df1.iterrows():
        q_name = row.iloc[1]  # Access QName from the second column (index 1)

        # Proceed only if QNAME exists in the second dataframe
        match_row = df2[df2.iloc[:, 1] == q_name]
        if not match_row.empty:
            df1_domain_text = str(row.iloc[column_index]).split('\n')
            df2_domain_text = str(match_row.iloc[0, column_index]).split('\n')
            
            # Print differences, now omitting QNames not found and formatting output
            differences_found = False
            for i, item in enumerate(df1_domain_text, start=1):
                match_item = df2_domain_text[i - 1] if i <= len(df2_domain_text) else 'MISSING'
                if item != match_item:
                    if not differences_found:
                        print(f"\nQName: {q_name}")
                        differences_found = True
                    print(f"{comparison_type} -> {item}")
                    print(f"Country -> {match_item}")
            if differences_found:
                print("\n-----\n")

# Load the template and country questionnaire files into pandas DataFrames
template_df = pd.read_excel(template_questionnaire_file, header=None, skiprows=2)
country_df = pd.read_excel(country_questionnaire_file, header=None, skiprows=2)

# English descriptions comparison (Column D is at index 3)
compare_and_print_differences(template_df, country_df, 3, "Template")
# Other language descriptions comparison (Column F is at index 5)
compare_and_print_differences(template_df, country_df, 5, "Template")

print("Comparison of Domains - Marked Change from Template - ENTIRE OUTPUT\n")

# Conditionally load and compare the previous round questionnaire
if previousround_questionnaire == "yes":
    previous_round_df = pd.read_excel(previousround_questionnaire_file, header=None, skiprows=2)
    print("Previous Round and Country Comparison:")
    compare_and_print_differences(previous_round_df, country_df, 3, "Previous Round")
    compare_and_print_differences(previous_round_df, country_df, 5, "Previous Round")

Comparison of Domains - Marked Change from Template


QName: NoAnswer
Template -> Phone rang and no one answered. Inclusive of voicemail, answering machine and temporary issues with network connection. 
Country -> Phone rang and no one answered.

-----


QName: NonWorkingNumber
Template -> Phone number is not in service, not working, or not valid.
Country -> Phone was disconnected or out of service.

-----


QName: resp_language
Template -> 2)other options [add as many as necessary]
Country -> 2)Shona

-----


QName: introduction
Template -> The survey usually takes 15/20 minutes to complete. Any information that you provide will be kept strictly confidential, only used for the purpose of the survey and will not be shown to other people. Your voluntary participation in the outcome of this interview is NOT IN ANY WAY linked to your personal chance of receiving any assistance. This is voluntary and you can choose not to answer any or all of the questions if you want. However, we hope tha

In [94]:
if previousround_questionnaire == "yes":
    
    # Importing the 'create_question_changes_sheet' function from the 'new_geopoll_functions' module
    from geopoll_modified import create_question_changes_sheet

    # Print a header message to indicate the start of the process
    print("CREATE OUTPUT VALIDATED QUESTIONNAIRE FILE")

    # Construct the path and filename for the validated questionnaire file. The filename 
    # includes the enumerator, language (lang), country code (iso3), and current timestamp (now)
    country_questionnaire_edited_file = os.path.join(temp_path, "validated_questionnaire_%s_%s_%s_%s.xlsx" % (enumerator, lang, iso3, now))

    # Copy the original questionnaire file to the newly constructed path
    copyfile(country_questionnaire_file, country_questionnaire_edited_file)

    # Construct a message indicating the path to the output validated country questionnaire
    message = "\n\nOutput validated country questionnaire: %s" % (country_questionnaire_edited_file)

    # Print the constructed message to the console
    print(message)

    # Append the constructed message to the 'execution_messages' string (assumed to be declared elsewhere in the code)
    execution_messages += message

    # Load the copied questionnaire file into an openpyxl workbook object
    wb = openpyxl.load_workbook(country_questionnaire_edited_file)

    # Call the 'create_question_changes_sheet' function to create or update a sheet named "Question changes" in the workbook
    #The function will use data from the unique questions in the questionnaire to populate this sheet
    #wb = create_question_changes_sheet(wb, unique_country_questionnaire_qname, unique_template_questionnaire_qname, unique_country_questionnaire, unique_template_questionnaire)

    wb = create_question_changes_sheet(wb, unique_country_questionnaire_qname, unique_previousround_questionnaire_qname, unique_country_questionnaire, unique_previousround_questionnaire)


    # Save the changes made to the workbook back to the same file path
    wb.save(country_questionnaire_edited_file)
    
else:
    print ("CREATE OUTPUT VALIDATED QUESTIONNAIRE FILE")
    country_questionnaire_edited_file = os.path.join(temp_path, "validated_questionnaire_%s_%s_%s_%s.xlsx" % (enumerator,lang,iso3, now)) 
    copyfile(country_questionnaire_file, country_questionnaire_edited_file)
    message =  "\n\nOutput validated country questionnaire: %s" % (country_questionnaire_edited_file)
    print(message)
    execution_messages += message

CREATE OUTPUT VALIDATED QUESTIONNAIRE FILE


Output validated country questionnaire: /Users/ibrahimaliahmad/Library/Mobile Documents/com~apple~CloudDocs/Documents/FAO/Data in Emergencies (DIEM)/Working Files/Questionnaire Validation/Script/questionnaire_validation/Questionnaire Validation/Questionnaires/Validated Questionnaires/ZWE/R11/V2/validated_questionnaire_geopoll_en_ZWE_20250718132411.xlsx


In [95]:
##CHECK THE QUESTIONNAIRE BEFORE PROCEEDING TO THE NEXT CELL##
##IF ALL CELLS ARE FILLED ON THE QUESTIONNAIRE TO BE VALIDATED##

# Display a message prompting the user to check the questionnaire before proceeding.
print("EDITING THE SURVEY BY REPLACING VALUES ACCORDING ADDITIONAL INFO SHEET")

# Check if the previous round's questionnaire was marked as "yes".
if previousround_questionnaire == "yes":
    # Update the questionnaire with values from the template using the country-specific information.
    #geopoll_modified.update_questionnaire(template_questionnaire_file, country_questionnaire_edited_file)
    geopoll_modified.update_questionnaire(template_questionnaire_file, country_questionnaire_edited_file)

# Search for and replace specific strings in the country's edited questionnaire.
geopoll_modified.find_and_replace_strings_in_df(country_questionnaire_edited_file)

# Add a message to the execution_messages indicating that the survey has been edited.
execution_messages += "\n\nSurvey edited by replacing values according to the Additional Information sheet"

EDITING THE SURVEY BY REPLACING VALUES ACCORDING ADDITIONAL INFO SHEET


In [96]:
# Printing a prompt message for the user
print("INSERT ADMIN SHEET WITH UPDATED BOUNDARIES")

# Calling the 'insert_sheet_with_adm_reference' function from 'geopoll_modified' module and storing the result in 'df'
df = geopoll_modified.insert_sheet_with_adm_reference(country_questionnaire_edited_file, admin_level, iso3)

# Displaying the first few rows of the dataframe 'df'
df.head()

# Appending a success message to the 'execution_messages' string
execution_messages += "\n\nCreated Admin Info sheet with updated boundaries."

INSERT ADMIN SHEET WITH UPDATED BOUNDARIES


In [100]:
# Print an announcement that we're beginning the edit process.
print("EDITING THE SURVEY BY SORTING THE CROP LIST VALUES ACCORDING TO PRIORITISE")

# Call the function to sort the crop list and get the number of most common choices.
n_of_choices = geopoll_modified.sort_crop_list_by_selection(country_questionnaire_edited_file)

# Format a message with the number of most common crops selected.
message13 = "\nNumber of most common crops selected in the Crop list sheet: %s" % n_of_choices
print(message13)

# Add a message indicating the editing process and the result to the `execution_messages`.
execution_messages += "\n\nSurvey edited by sorting the crop list according to the prioritised crops" + message13

EDITING THE SURVEY BY SORTING THE CROP LIST VALUES ACCORDING TO PRIORITISE

Number of most common crops selected in the Crop list sheet: 10


In [105]:
print("Create output file with execution messages")
text_file = os.path.join(temp_path, "questionnaire_validation_report_%s_R%s_%s.txt" % (iso3,round_number,now)) 
file = open(text_file, "w") 
file.write(execution_messages) 
file.close() 

Create output file with execution messages


In [107]:
print("Execution time: ", datetime.now() - startTime)

Execution time:  0:00:10.332102
